# 03 — Train Emulators (Multi-z): Cluster Profiles

All cluster profile statistics: CGD, CGED, CPP, CTP, CEP, CEEP, CMP, CYP.
Multi-snapshot emulators (z ≤ 0.5, snapshots 415–624).
All models saved to `../models/<PROFILE>_multiz/`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import matplotlib.cm as cm
import os

from cosmo_hydro_emu.pca import *
from cosmo_hydro_emu.viz import *
from cosmo_hydro_emu.load_hacc import *
from cosmo_hydro_emu.emu import *
from cosmo_hydro_emu.gp import *
from cosmo_hydro_emu.snapshot_utils import SNAPSHOT_IDS, get_snapshot_redshifts

## Configuration

In [ ]:
DirIn = '../data/400MPC_RUNS_5SG_2COSMO_PARAM/HAvoCC/'

start_sim_idx = 1
num_sims = 39
exp_variance = 0.999

z_initial = 200

do_train = True

seed_mass_scale = 1e6
vkin_scale = 1e4
eps_scale = 1e1

## Load parameters

In [ ]:
fileIn = '../data/FinalDesign.txt'
params_all = np.loadtxt(fileIn, delimiter=",", skiprows=1)
# Design CSV row K corresponds to RUN_K (both 0-indexed); row 0 = RUN000.
# With start_sim_idx=1, num_sims=39 the loaders read RUN001..RUN039; the
# matching slice is rows [1:40] = [start_sim_idx : start_sim_idx + num_sims].
params32 = params_all[start_sim_idx : start_sim_idx + num_sims]
params32[:, 2] /= seed_mass_scale
params32[:, 3] /= vkin_scale
params32[:, 4] /= eps_scale

print('params32 shape:', params32.shape)

# Train/test split
test_sim_indices = np.array([3, 11, 19, 27, 35])
train_sim_indices = np.array([i for i in range(num_sims) if i not in test_sim_indices])

params_train = params32[train_sim_indices]
params_test = params32[test_sim_indices]

print(f'Train: {len(train_sim_indices)} sims, Test: {len(test_sim_indices)} sims')

## Snapshot setup

In [ ]:
z_all, a_all = get_snapshot_redshifts(SNAPSHOT_IDS, z_initial=z_initial)

print(f'Number of snapshots: {len(SNAPSHOT_IDS)}')
print(f'Snapshot IDs: {SNAPSHOT_IDS}')
print(f'Redshift range: z = {z_all[-1]:.2f} to {z_all[0]:.2f}')
print(f'Scale factor range: a = {a_all[0]:.3f} to {a_all[-1]:.3f}')

Number of snapshots: 11
Snapshot IDs: [205, 224, 247, 275, 310, 355, 415, 479, 498, 567, 624]
Redshift range: z = 0.00 to 2.00
Scale factor range: a = 0.333 to 1.000


## Load all profile data

In [ ]:
from cosmo_hydro_emu.load_hacc import read_profile_all_snaps

PROFILE_CONFIGS = {
    'CGD':  {'file_prefix': 'ClusterGasDensityProfile',         'label': 'Cluster Gas Density'},
    'CGED': {'file_prefix': 'ClusterGasElectronDensityProfile',  'label': 'Cluster Gas Electron Density'},
    'CPP':  {'file_prefix': 'ClusterGasPressureProfile',         'label': 'Cluster Gas Pressure'},
    'CTP':  {'file_prefix': 'ClusterGasTemperatureProfile',      'label': 'Cluster Gas Temperature'},
    'CEP':  {'file_prefix': 'ClusterGasEntropyProfile',          'label': 'Cluster Gas Entropy'},
    'CEEP': {'file_prefix': 'ClusterElectronEntropyProfile',     'label': 'Cluster Electron Entropy'},
    'CMP':  {'file_prefix': 'ClusterGasMetallicityProfile',      'label': 'Cluster Gas Metallicity'},
    'CYP':  {'file_prefix': 'ClusterGasYProfile',                'label': 'Cluster Compton-y (tSZ)'},
}

profile_data = {}
for short_name, config in PROFILE_CONFIGS.items():
    radius, arr = read_profile_all_snaps(DirIn, num_sims, SNAPSHOT_IDS,
                                          config['file_prefix'],
                                          start_sim_idx=start_sim_idx)
    profile_data[short_name] = arr
    print(f"{short_name}: {arr.shape}")

print(f"Radius bins: {radius.shape}")

CGD: (39, 11, 19)
CGED: (39, 11, 19)
CPP: (39, 11, 19)
CTP: (39, 11, 19)
CEP: (39, 11, 19)
CEEP: (39, 11, 19)
CMP: (39, 11, 19)
CYP: (39, 11, 19)
Radius bins: (19,)


## Radius cut

In [ ]:
rlim1, rlim2 = mass_conds('CGD')  # same limits for all profiles
rad_cond = np.where((radius > rlim1) & (radius < rlim2))[0]
y_ind_profiles = radius[rad_cond]
print(f"Radius cut: {len(rad_cond)} bins in [{rlim1:.3f}, {rlim2:.3f}]")

Radius cut: 19 bins in [0.015, 2.750]


## Train all profiles

In [ ]:
profile_z_start_idx = 6  # z <= 0.5
z_index_range = np.arange(profile_z_start_idx, len(SNAPSHOT_IDS))
profile_z_all = z_all[profile_z_start_idx:]

profile_models = {}

for short_name, config in PROFILE_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Training {short_name}: {config['label']}")
    print(f"{'='*60}")

    arr = profile_data[short_name]
    y_vals = arr[:, :, rad_cond]  # (num_sims, num_snaps, n_bins_cut)

    model_dir = f'../models/{short_name}_multiz/'

    if do_train:
        os.makedirs(model_dir, exist_ok=True)
        do_gp_train_multiple(
            model_dir=model_dir,
            p_train_all=params_train,
            y_vals_all=y_vals[train_sim_indices],
            y_ind_all=y_ind_profiles,
            z_index_range=z_index_range,
            exp_variance=exp_variance
        )

    model_list, data_list = load_model_multiple(
        model_dir=model_dir,
        p_train_all=params_train,
        y_vals_all=y_vals[train_sim_indices],
        y_ind_all=y_ind_profiles,
        z_index_range=z_index_range,
    )

    profile_models[short_name] = (model_list, data_list)
    print(f"  Loaded {len(model_list)} models for {short_name}")


Training CGD: Cluster Gas Density
Starting tune_step_sizes...
Default step sizes:
betaU
[[0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]]
lamUz
[[5. 5. 5. 5. 5.]]
lamWs
[[100. 100. 100. 100. 100.]]
lamWOs
[[100.]]


Step size tuning: 100%|██████████| 50/50 [00:37<00:00,  1.32it/s]


Done with tune_step_size.
Selected step sizes:
betaU
[[0.27685039 0.13290753 0.3002617  0.72648201 0.82648575]
 [2.56110182 3.46787087 1.17191939 2.90720803 1.78109237]
 [4.458669   4.20830148 2.46387537 4.89932259 1.15084493]
 [1.75670513 1.59816544 0.74748805 1.79119221 6.0082047 ]
 [1.37142363 2.557543   6.94909258 3.5264182  2.14342037]
 [4.13151543 1.78832962 5.48420107 2.21249918 1.20799906]
 [2.26334425 2.85214047 0.43255176 2.69993893 2.38044209]
 [1.35531903 3.01756999 1.77936391 4.20454645 6.03713428]]
lamUz
[[1.68805054 1.76748209 1.64960205 2.01503029 1.78839275]]
lamWs
[[4074.98514941 5870.89667716 4838.09436327 4088.37703158 6293.58757491]]
lamWOs
[[818.2209293]]


MCMC sampling: 100%|██████████| 1000/1000 [00:23<00:00, 42.16it/s]


Model saved to ../models/CGD_multiz/multivariate_model_z_index6.pkl
Training complete for snapshot 6
Model saved at ../models/CGD_multiz/
=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*
Starting tune_step_sizes...
Default step sizes:
betaU
[[0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1]]
lamUz
[[5. 5. 5. 5. 5.]]
lamWs
[[100. 100. 100. 100. 100.]]
lamWOs
[[100.]]


...

MCMC sampling:   0%|          | 0/1000 [00:00<?, ?it/s]

Done with tune_step_size.
Selected step sizes:
betaU
[[0.12561119 0.189552   0.25985107 0.2033575  1.04227556 0.84643121
  0.31631562 1.36413752 0.29980023 0.38652776]
 [0.75479756 4.20860753 4.13579674 3.93935391 4.25724326 0.72521415
  0.05913118 3.31603095 1.60260406 2.07973245]
 [3.95189651 9.76953714 0.59660497 4.40931612 1.28434039 0.30955447
  1.2913631  5.14203008 0.20794141 0.77725904]
 [5.01722907 1.48910963 0.31108498 4.34268934 4.30010313 3.02674637
  5.8717948  1.80639819 1.92242837 1.04347843]
 [2.74011708 2.16694741 3.46575642 2.01527117 3.09530322 2.21185686
  1.76158251 1.99678129 2.7612738  0.49707151]
 [1.64163741 4.74111531 2.52624357 0.54301962 0.25094776 2.72750098
  1.59501306 4.98215108 0.40820232 0.26607846]
 [3.23224502 6.51807851 8.25314755 1.26003143 2.27421306 0.95417851
  0.53463419 4.9568081  0.33836868 1.23834722]
 [2.3381521  2.2259909  2.42952762 1.20744634 2.72069654 0.80111499
  0.07996591 1.64079478 2.71186289 1.44046232]]
lamUz
[[1.28725312 1.60295

Step size tuning:   0%|          | 0/50 [00:00<?, ?it/s]

Model saved to ../models/CMP_multiz/multivariate_model_z_index9.pkl
Training complete for snapshot 9
Model saved at ../models/CMP_multiz/
=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*=*
Starting tune_step_sizes...
Default step sizes:
betaU
[[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]
 [0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1]]
lamUz
[[5. 5. 5. 5. 5. 5. 5. 5. 5.]]
lamWs
[[100. 100. 100. 100. 100. 100. 100. 100. 100.]]
lamWOs
[[100.]]


Step size tuning:  22%|██▏       | 11/50 [00:10<00:35,  1.09it/s]

## Validation at z=0 (all profiles)

In [ ]:
n_profiles = len(PROFILE_CONFIGS)
ncols = 4
nrows = (n_profiles + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows))
axes = axes.flatten()

for idx, (short_name, config) in enumerate(PROFILE_CONFIGS.items()):
    ax = axes[idx]
    model_list, data_list = profile_models[short_name]
    arr = profile_data[short_name]
    y_vals = arr[:, :, rad_cond]

    for t_idx in test_sim_indices[:3]:
        target = y_vals[t_idx, -1, :]  # z=0 (last snapshot)
        pred_mean, pred_quant = emulate(model_list[-1], params32[t_idx])
        ax.plot(y_ind_profiles, target, 'k-', alpha=0.5)
        ax.plot(y_ind_profiles, pred_mean[:, 0], 'r--', alpha=0.7)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(short_name)
    ax.set_xlabel(r'$r/R_{500c}$')

for idx in range(len(PROFILE_CONFIGS), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('../plots/profiles_multiz_validation.png', bbox_inches='tight')

## Redshift interpolation test

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

test_params = params32[test_sim_indices[0]]
z_grid = np.linspace(profile_z_all[-2], profile_z_all[1], 5)

for idx, (short_name, config) in enumerate(PROFILE_CONFIGS.items()):
    ax = axes[idx]
    model_list, data_list = profile_models[short_name]

    for z_target in z_grid:
        params_with_z = np.append(test_params, [z_target])[np.newaxis, :]
        pred_z, pred_z_err = emu_redshift(params_with_z, model_list, data_list, profile_z_all)
        ax.plot(y_ind_profiles, pred_z[:, 0], label=f'z={z_target:.2f}')

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(short_name)
    ax.legend(fontsize=6)

plt.tight_layout()
plt.savefig('../plots/profiles_multiz_redshift_interp.png', bbox_inches='tight')